# balance-beam: calibration data analysis

This notebook evaluates the accuracy of the balance-beam scale using the 26-trial calibration dataset in `data/calibration-data.csv`. Each trial pairs a known mass with the voltage the sensor produced.

**What it does**

1. Loads the dataset.
2. Applies the same two-point model that runs on the ESP32 (`mass = voltage x 1000 / 1.859`).
3. Reports error metrics.
4. Plots the calibration curve and the error at each mass.

**How to run it**

- Locally: `pip install -r requirements.txt`, then `jupyter notebook notebooks/calibration-analysis.ipynb`.
- In Colab: use the badge in the README. The notebook downloads the dataset from GitHub if it cannot find it locally.

Use **Run All** and keep the outputs when you save, so the results show up on GitHub.

## 1. Setup

The two constants below come from the firmware (`src/main.py`): the scale reads **1.859 V** with the 1000 g calibration weight on the pan, and the device converts voltage to grams proportionally.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FULL_SCALE_V = 1.859    # volts measured at 1000 g (see src/main.py)
FULL_SCALE_G = 1000.0   # calibration weight in grams

LOCAL_PATHS = [Path("../data/calibration-data.csv"), Path("data/calibration-data.csv")]
RAW_URL = "https://raw.githubusercontent.com/NehaM2025/balance-beam/main/data/calibration-data.csv"

## 2. Load the dataset

The notebook looks for the CSV next to the repository first, then falls back to downloading it from GitHub.

In [ ]:
def load_data():
    for path in LOCAL_PATHS:
        if path.exists():
            print(f"Loaded {path}")
            return pd.read_csv(path)
    print("Local file not found. Downloading from GitHub.")
    return pd.read_csv(RAW_URL)


df = load_data()
print(f"{len(df)} rows, columns: {list(df.columns)}")
df.head()

## 3. Identify the mass and voltage columns

The notebook finds the columns whose names contain "mass" and "volt". If your header uses different names, set `MASS_COL` and `VOLT_COL` by hand.

In [ ]:
MASS_COL = None   # for example "mass_g"
VOLT_COL = None   # for example "voltage_v"


def find_column(frame, keyword):
    matches = [c for c in frame.columns if keyword in c.lower()]
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one column containing '{keyword}', found {matches}. "
            f"Columns are {list(frame.columns)}. Set MASS_COL and VOLT_COL by hand."
        )
    return matches[0]


MASS_COL = MASS_COL or find_column(df, "mass")
VOLT_COL = VOLT_COL or find_column(df, "volt")

data = (
    df[[MASS_COL, VOLT_COL]]
    .rename(columns={MASS_COL: "mass_g", VOLT_COL: "volts"})
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
    .reset_index(drop=True)
)
print(f"Using {len(data)} trials: mass from '{MASS_COL}', voltage from '{VOLT_COL}'")
data.describe()

## 4. Apply the on-device model

The firmware stores the ADC reading at 1000 g and scales every later reading proportionally. In voltage terms, that is `mass = voltage x 1000 / 1.859`. Relative error is undefined at 0 g, so the error metrics use the non-zero trials only.

In [ ]:
data["predicted_g"] = data["volts"] * FULL_SCALE_G / FULL_SCALE_V
data["error_g"] = data["predicted_g"] - data["mass_g"]
data["abs_error_g"] = data["error_g"].abs()

nonzero = data[data["mass_g"] > 0].copy()
nonzero["rel_error"] = nonzero["abs_error_g"] / nonzero["mass_g"]

data.head(10)

## 5. Error metrics

These are the numbers reported in the README's Results table. Relative error is measured against the known mass.

In [ ]:
heavy = nonzero[nonzero["mass_g"] >= 400]
worst = nonzero.loc[nonzero["abs_error_g"].idxmax()]

# R-squared of the least-squares line of voltage against mass, using every trial
slope, intercept = np.polyfit(data["mass_g"], data["volts"], 1)
fitted = slope * data["mass_g"] + intercept
ss_res = ((data["volts"] - fitted) ** 2).sum()
ss_tot = ((data["volts"] - data["volts"].mean()) ** 2).sum()
r_squared = 1 - ss_res / ss_tot

metrics = pd.Series({
    "Non-zero trials": len(nonzero),
    "Median absolute error (g)": round(nonzero["abs_error_g"].median(), 1),
    "Trials within 30 g (3% of full scale)": int((nonzero["abs_error_g"] <= 30).sum()),
    "Trials within 3% of the known mass": int((nonzero["rel_error"] <= 0.03).sum()),
    "Mean relative error, masses >= 400 g (%)": round(heavy["rel_error"].mean() * 100, 1),
    "Worst single error (g)": round(worst["abs_error_g"], 1),
    "Mass of the worst trial (g)": worst["mass_g"],
    "R-squared (voltage vs. mass)": round(r_squared, 3),
}, dtype=object)
metrics

## 6. Calibration curve

Voltage should rise almost proportionally with mass. The dashed line is the on-device model, and the solid line is the least-squares fit of the data.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(data["mass_g"], data["volts"], label="Measured trials", zorder=3)

grid = np.linspace(0, FULL_SCALE_G, 100)
ax.plot(grid, grid * FULL_SCALE_V / FULL_SCALE_G, "--", label="On-device model (0 g and 1000 g)")
ax.plot(grid, slope * grid + intercept, label=f"Least-squares fit (R\u00b2 = {r_squared:.3f})")

ax.set_xlabel("Known mass (g)")
ax.set_ylabel("Sensor voltage (V)")
ax.set_title("Calibration curve")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 7. Error at each mass

Positive values mean the device read too high, and negative values mean it read too low. Errors cluster at low mass, which is why a multi-point calibration is the first next step in the README.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.axhline(0, color="gray", linewidth=1)
ax.bar(nonzero["mass_g"].astype(str), nonzero["error_g"].values)
ax.set_xlabel("Known mass (g)")
ax.set_ylabel("Error: predicted minus known (g)")
ax.set_title("Error of the on-device model by mass")
ax.tick_params(axis="x", rotation=60)
ax.grid(axis="y", alpha=0.3)
plt.show()

## 8. What to try next

- Fit a multi-point calibration (for example 0, 200, 500, and 1000 g) and compare its error with the two-point model above.
- Take the median of the 25 readings instead of the mean and check whether the outliers shrink.
- Add temperature or drift columns to the dataset to see whether they explain the low-mass error.